In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer
)

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import evaluate

/users/eleves-b/2022/louis.martino/TA_Group-Project/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_feature_names(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

FEATURES_NO_SCALE = load_feature_names("../data/features/features_no_scale.txt")
FEATURES_SCALE = load_feature_names("../data/features/features_scale.txt")

print("No-scale features:", len(FEATURES_NO_SCALE))
print("Scale features:", len(FEATURES_SCALE))

No-scale features: 21
Scale features: 8


In [7]:
train_raw = pd.read_csv("../data/features/train.csv")
val_raw = pd.read_csv("../data/features/val.csv")

print("Train raw shape:", train_raw.shape)
print("Val raw shape:", val_raw.shape)

Train raw shape: (66096, 42)
Val raw shape: (22032, 42)


In [8]:
def prepare_aligned(df):
    df = df[df["state"].isin(["successful", "failed"])].copy()
    df["text"] = df["name"].fillna("") + " " + df["blurb"].fillna("")
    df["label"] = (df["state"] == "successful").astype(int)

    needed_cols = ["text", "label"] + FEATURES_NO_SCALE + FEATURES_SCALE
    df = df.dropna(subset=["text", "label"]).copy()

    # keep only rows where structured features exist
    df = df.dropna(subset=FEATURES_NO_SCALE + FEATURES_SCALE).reset_index(drop=True)

    return df[needed_cols]

train_df = prepare_aligned(train_raw)
val_df = prepare_aligned(val_raw)

print("Train prepared shape:", train_df.shape)
print("Val prepared shape:", val_df.shape)
train_df.head()

Train prepared shape: (66096, 31)
Val prepared shape: (22032, 31)


,text,label,cat_Art,cat_Comics,cat_Crafts,cat_Dance,cat_Design,cat_Fashion,cat_Film & Video,cat_Food,...,country_US,z-score_log_goal,duration,CCI_index,blurb_length,sentiment_score,readability_score,name_blurb_similarity,log_goal,CCI_per_goal
0,Bring Art ala Carte back to the community! Art...,0,1,0,0,0,0,0,0,0,...,1,1.150449,30,101.23980,21,0.4738,71.291786,0.906809,10.126671,0.004050
1,The PRESENT MIND Deck Mentalist Grant Price de...,1,0,0,0,0,0,0,0,0,...,1,0.750864,28,98.42246,22,0.7650,75.320357,0.770244,9.105091,0.010936
2,Zeus-X Pro ⚡ - The World's Most Futuristic Uni...,1,0,0,0,0,0,0,0,0,...,0,-0.599290,35,99.30466,22,0.6221,76.460909,0.737919,8.382949,0.022720
3,Make 100 : The Spotting Chart Project - WWII A...,1,0,0,0,0,1,0,0,0,...,1,-0.649362,21,101.38930,20,0.3612,59.635000,0.720486,6.685861,0.126737
4,Art & Hue | Graphic Pop Art | Bespoke & Custom...,1,1,0,0,0,0,0,0,0,...,0,0.641695,30,101.81250,24,0.2500,66.423370,0.789146,8.353842,0.023981


In [9]:
scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df[FEATURES_SCALE].astype(np.float32))
val_scaled = scaler.transform(val_df[FEATURES_SCALE].astype(np.float32))

train_no_scale = train_df[FEATURES_NO_SCALE].astype(np.float32).to_numpy()
val_no_scale = val_df[FEATURES_NO_SCALE].astype(np.float32).to_numpy()

X_train_struct = np.hstack([train_no_scale, train_scaled]).astype(np.float32)
X_val_struct = np.hstack([val_no_scale, val_scaled]).astype(np.float32)

X_train_struct = np.nan_to_num(X_train_struct, nan=0.0, posinf=0.0, neginf=0.0)
X_val_struct = np.nan_to_num(X_val_struct, nan=0.0, posinf=0.0, neginf=0.0)

print("Structured train shape:", X_train_struct.shape)
print("Structured val shape:", X_val_struct.shape)

Structured train shape: (66096, 29)
Structured val shape: (22032, 29)


In [10]:
train_text_df = pd.DataFrame({
    "text": train_df["text"].values,
    "label": train_df["label"].values
})

val_text_df = pd.DataFrame({
    "text": val_df["text"].values,
    "label": val_df["label"].values
})

train_dataset = Dataset.from_pandas(train_text_df)
val_dataset = Dataset.from_pandas(val_text_df)

In [11]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [12]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 22032/22032 [00:00<00:00, 28935.19 examples/s]


In [13]:
train_dataset = train_dataset.add_column("structured_features", X_train_struct.tolist())
val_dataset = val_dataset.add_column("structured_features", X_val_struct.tolist())

In [14]:
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "structured_features", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "structured_features", "label"]
)

print(train_dataset[0].keys())
print(train_dataset[0]["structured_features"].shape)

dict_keys(['label', 'input_ids', 'attention_mask', 'structured_features'])
torch.Size([29])


In [22]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
roc_auc_metric = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels)
    recall = recall_metric.compute(predictions=predictions, references=labels)
    roc_auc = roc_auc_metric.compute(prediction_scores=probs, references=labels)

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "roc_auc": roc_auc["roc_auc"],
    }

In [16]:
class DistilBERTWithStructuredFeatures(nn.Module):
    def __init__(self, model_name, structured_dim, num_labels=2, dropout=0.2, hidden_dim=256):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)

        text_hidden_size = self.text_encoder.config.hidden_size  # 768 for DistilBERT

        self.classifier = nn.Sequential(
            nn.Linear(text_hidden_size + structured_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels)
        )

        self.loss_fn = nn.CrossEntropyLoss()

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        masked_embeddings = last_hidden_state * mask
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids=None, attention_mask=None, structured_features=None, labels=None):
        outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        last_hidden_state = outputs.last_hidden_state
        pooled_text = self.mean_pooling(last_hidden_state, attention_mask)
        pooled_text = self.dropout(pooled_text)

        combined = torch.cat([pooled_text, structured_features], dim=1)
        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [17]:
structured_dim = X_train_struct.shape[1]

model = DistilBERTWithStructuredFeatures(
    model_name=model_name,
    structured_dim=structured_dim,
    num_labels=2,
    dropout=0.2,
    hidden_dim=256
)

print("Structured dim:", structured_dim)

Structured dim: 29


In [18]:
training_args = TrainingArguments(
    output_dir="./joint_distilbert_structured_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    report_to="none"
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_276825/2721217396.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.500000,0.461934,0.782589,0.821415,0.800931,0.842975
2,0.384400,0.506303,0.782453,0.823677,0.793128,0.856673


TrainOutput(global_step=16524, training_loss=0.44223173903772545, metrics={'train_runtime': 1375.0055, 'train_samples_per_second': 96.139, 'train_steps_per_second': 12.017, 'total_flos': 0.0, 'train_loss': 0.44223173903772545, 'epoch': 2.0})

In [23]:
val_results = trainer.evaluate()
val_results

{'eval_loss': 0.5063025951385498,
 'eval_accuracy': 0.7824527959331881,
 'eval_f1': 0.8236765625574808,
 'eval_precision': 0.7931278781438187,
 'eval_recall': 0.8566727884909703,
 'eval_runtime': 68.5274,
 'eval_samples_per_second': 321.506,
 'eval_steps_per_second': 40.188,
 'epoch': 2.0}

In [24]:
pred_output = trainer.predict(val_dataset)
preds = np.argmax(pred_output.predictions, axis=-1)

print(preds[:10])

[1 0 0 1 1 1 1 1 1 0]
